## Part 1: K-Nearest Neighbor

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.datasets import make_moons
# Generate the dataset (everyone uses the same seed for consistency)
np.random.seed(42)
X, y = make_moons(n_samples=200, noise=0.15)

In [2]:
# Implement your own KNN classification algorithm

def knn_predict(X_train, y_train, X_test, y_test, k, distance_func):
    predictions = []
    
    for x in X_test:
        # Use Manhattan Distance
        if distance_func == 1:
            distances = np.sum(np.abs(X_train - x), axis = 1)
        # Use Euclidian Distance
        elif distance_func == 2:
            distances = np.sqrt(np.sum((X_train - x) ** 2, axis = 1))
        else:
            raise ValueError("distance_func must be either 1 (Manhattan Distance) or 2 (Euclidian distance)")
        
        # Find k nearest indices
        nearest_indices = np.argsort(distances)[:k]
        # Find k nearest labels
        nearest_labels = y_train[nearest_indices]
        # Find k nearest distances
        nearest_distances = distances[nearest_indices]
        # Find winning class
        counts = Counter(nearest_labels)
        max_count = max(counts.values())
        
        tied_classes = [label for label, count in counts.items()
                       if count == max_count]
        
        # There is no tie, return prediction
        if len(tied_classes) == 1:
            prediction = tied_classes[0]
            
        # If tied, resolve with weighting based on distance
        else:
            weights = {}
        
            for label in tied_classes:
                class_distances = nearest_distances[nearest_labels == label]
                weights[label] = np.sum(1 / (class_distances + 1e-7))
            
            max_weight = max(weights.values())
            
            weighted_ties = [label for label, weight in weights.items()
                            if weight == max_weight]
            
            # There is no tie after weighting, return prediction
            if len(weighted_ties) == 1:
                prediction = weighted_ties[0]
               
            # If a tie remains, pick random class
            else:
                prediction = np.random.choice(weighted_ties)
                
        predictions.append(prediction)
    
    # Return all predictions made
    predictions = np.array(predictions)
    # Calculate accuracy based on predictions and true values
    if y_test != None:
        accuracy = np.mean(predictions == y_test)
    else:
        accuracy = None
    
    return predictions, accuracy
              

In [3]:
# Implement a function that visualizes a 2-D KNN model

import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

def visualize_knn(X_train, y_train, k, distance_func):
    # Get minimium and maximum values
    x_min, x_max = X_train[:, 0].min() - 1, X_train[:, 0].max() + 1
    y_min, y_max = X_train[:, 1].min() - 1, X_train[:, 1].max() + 1
    
    # Randomly generate 10,000 points
    X_grid = np.column_stack(
        np.random.uniform(x_min, x_max, 10000),
        np.random.uniform(y_min, y_max, 10000)
    )
    
    # Predict the classes of the random points using KNN
    predictions, accuracy = knn_predict(X_train, y_train, X_grid, None, k, distance_func)
    
    # Plot the predicted classes
    for label in np.unique(predictions):
        plt.scatter(
            X_grid[predictions == label, 0],
            X_grid[predictions == label, 1],
            label=f"Predicted {label}",
            alpha=0.15
        )
        
    # Plot the training points, colored by their actual classes
    for label in np.unique(y_train):
        plt.scatter(
            X_train[y_train == label, 0],
            X_train[y_train == label, 1],
            edgecolors="black",
            label=f"Actual {label}"
        )
    
    # Add plot labels and legend
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.title(f"KNN Decision Boundary (k={k})")
    plt.legend()
    plt.show()


In [4]:
# Implement function that computes training and test error for a k-NN model

def knn_error(X_train, y_train, k, distance_func):
    # Training error: use all training records
    train_predictions, _ = knn_predict(X_train, y_train, X_train, y_train, k, distance_func)
    
    # Calculate training error
    train_error = np.mean(train_predictions != y_train)
    
    # LOOCV: remove the point being predicted (test record), use all others for training
    loocv_predictions = []
    
    for i in range(len(X_train)):
        X_fold = np.delete(X_train, i, axis=0)
        y_fold = np.delete(y_train, i, axis=0)
        
        prediction, _ = knn(predict(X_fold, y_fold, X_train[i:i+1], y_train[i:i+1], k, distance_func))
        
        loocv_predictions.append(prediction[0])
    
    # Calculate testing error
    loocv_error = np.mean(np.array(loocv_predictions) != y_train)
                            
    return train_error, loocv_error

## Part 2: Neural Network

In [6]:
# Implement perceptron training rule

def train_perceptron(training_record, learning_rate, w1, w2, b):
    # Training record is in schema (x1, x2, y)
    x1, x2, y = training_record
    
    # Calculate perceptron output
    z = (w1 * x1) + (w2 * x2) + b
    prediction = 1 if z >= 0 else 0
    
    # Calulate prediction error
    error = y - prediction
    
    # Update weights and bias
    w1 = w1 + learning_rate * error * x1
    w2 = w2 + learning_rate * error * x2
    b = b + learning_rate * error
    
    return w1, w2, b

In [1]:
# Implement single-unit neural network training

def train_neuron(training_record, learning_rate, w, b, cost_derivative, activation_derivative):
    # Training record has schema (x, y)
    x, y = training_record
    
    # Calculate neuron output
    z = w*x + b

    # Activation functions
    # Sigmoid function
    if activation_derivative == 1:
        a = 1 / (1 + np.exp(-z))
    # Tanh
    elif activation_derivative == 2:
        a = np.tanh(z)
    # ReLU
    elif activation_derivative == 3:
        a = max(0, z) 
    else:
        raise ValueError("activation_derivative must be 1 (Sigmoid), 2 (Tanh), or 3 (ReLU)")
        
    # Cost function derivatives
    # Squared error derivative
    if cost_derivative == 1:
        dC_da = a - y 
    # Binary cross entropy derivative
    elif cost_derivative == 2:
        dC_da = (a - y) / (a * (1-a))
    else:
        raise ValueError("cost_derivative must be 1 (squared error) or 2 (cross-entropy)")
    
    # Calculate activation derivative with respect to z
    # Sigmoid derivative
    if activation_derivative == 1:
        da_dz = a*(1-a)
    # Tanh derivative
    elif activation_derivative == 2:
        da_dz = 1 - a ** 2
    # ReLU derivative
    else:
        da_dz = 1 if z > 0 else 0
    
    # Calculate gradient with chain rule
    gradient = dC_da * da_dz
    
    # Update weight and bias using gradient descent
    w = w - (learning_rate * gradient * x)
    b = b - (learning_rate * gradient)
    
    return w, b
    